# Prueba - Optimización y despliegue de modelos predictivos supervisados

Curso: Fundamentos de Ciencia de Datos (SENCE / Desafío Latam)

Cierre del módulo: se construyen y optimizan (con `GridSearchCV`) dos pipelines independientes para una plataforma de formación continua — un clasificador que predice si un estudiante completará un curso (`completo`) y un regresor que estima cuántos cursos tomará el próximo trimestre (`cursos_futuros`).

## 0. Dataset

El archivo de apoyo entrega un dataset de ejemplo de 10 filas con las mismas columnas usadas en el desafío anterior (`edad`, `nivel_participacion`, `tiempo_en_plataforma`, `tipo_inscripcion`, `region`, `completo`, `cursos_futuros`). Nuevamente, para que `GridSearchCV` + `cross_validate` con particiones (`cv=5`) sean estadísticamente razonables, se genera un dataset simulado de 300 estudiantes con la misma estructura y relaciones realistas.

In [1]:
import pandas as pd
import numpy as np

np.random.seed(123)
n = 300

edad = np.random.randint(18, 55, size=n)
nivel_participacion = np.random.beta(2, 2, size=n).round(2)
tiempo_en_plataforma = (nivel_participacion * 20 + np.random.normal(0, 2, size=n)).clip(0, None).round(1)
tipo_inscripcion = np.random.choice(['Libre', 'Premium'], size=n, p=[0.55, 0.45])
region = np.random.choice(['Norte', 'Centro', 'Sur'], size=n, p=[0.3, 0.45, 0.25])

bonus_premium = np.where(tipo_inscripcion == 'Premium', 0.5, 0.0)
score = -1.5 + 3.5 * nivel_participacion + bonus_premium + np.random.normal(0, 0.4, size=n)
prob_completo = 1 / (1 + np.exp(-score))
completo = np.random.binomial(1, prob_completo)

cursos_futuros = np.round(
    np.clip(nivel_participacion * 5 + bonus_premium + np.random.normal(0, 0.7, size=n), 0, None)
).astype(int)

df = pd.DataFrame({
    'edad': edad,
    'nivel_participacion': nivel_participacion,
    'tiempo_en_plataforma': tiempo_en_plataforma,
    'tipo_inscripcion': tipo_inscripcion,
    'region': region,
    'completo': completo,
    'cursos_futuros': cursos_futuros,
})

print(f'Dimensiones: {df.shape}')
print(f'Balance de clases (completo):\n{df["completo"].value_counts(normalize=True)}')
df.head()

Dimensiones: (300, 7)
Balance de clases (completo):
completo
1    0.6
0    0.4
Name: proportion, dtype: float64


   edad  nivel_participacion  tiempo_en_plataforma tipo_inscripcion  region  \
0    20                 0.63                  16.0          Premium   Norte   
1    46                 0.35                   5.7            Libre   Norte   
2    52                 0.21                   0.4            Libre     Sur   
3    35                 0.04                   1.7            Libre   Norte   
4    37                 0.16                   5.2          Premium  Centro   

   completo  cursos_futuros  
0         1               4  
1         0               2  
2         0               0  
3         0               0  
4         0               3  

In [2]:
num_cols = ['edad', 'nivel_participacion', 'tiempo_en_plataforma']
cat_cols = ['tipo_inscripcion', 'region']

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import cross_validate, GridSearchCV, StratifiedKFold

preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),
])

## 1. Modelo de clasificación con búsqueda de hiperparámetros (`completo`)

Se usa `RandomForestClassifier`, optimizando `n_estimators` y `max_depth` con `GridSearchCV`. Como el dataset presenta cierto desbalance de clases, se usa `StratifiedKFold` tanto dentro de `GridSearchCV` como en la validación cruzada final.

In [3]:
from sklearn.ensemble import RandomForestClassifier

X_clf = df.drop(columns=['completo', 'cursos_futuros'])
y_clf = df['completo']

pipe_clf = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42)),
])

param_grid_clf = {
    'classifier__n_estimators': [100, 200],
    'classifier__max_depth': [3, 5, None],
}

cv_estratificado = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid_clf = GridSearchCV(
    pipe_clf, param_grid_clf, cv=cv_estratificado,
    scoring='f1_macro', n_jobs=-1,
)
grid_clf.fit(X_clf, y_clf)

print(f'Mejores parámetros (clasificación): {grid_clf.best_params_}')
print(f'Mejor f1_macro en la búsqueda: {grid_clf.best_score_:.4f}')

Mejores parámetros (clasificación): {'classifier__max_depth': 5, 'classifier__n_estimators': 100}
Mejor f1_macro en la búsqueda: 0.6251


In [4]:
resultados_clf = cross_validate(
    grid_clf.best_estimator_, X_clf, y_clf, cv=cv_estratificado,
    scoring=['accuracy', 'f1_macro'],
)

accuracy_promedio = resultados_clf['test_accuracy'].mean()
f1_macro_promedio = resultados_clf['test_f1_macro'].mean()

print(f'Accuracy promedio (modelo optimizado): {accuracy_promedio:.4f}')
print(f'F1_macro promedio (modelo optimizado): {f1_macro_promedio:.4f}')

Accuracy promedio (modelo optimizado): 0.6567
F1_macro promedio (modelo optimizado): 0.6251


**Interpretación técnica:** `GridSearchCV` probó combinaciones de `n_estimators` (cantidad de árboles) y `max_depth` (profundidad máxima) usando `f1_macro` como criterio de selección, apropiado dado el desbalance de clases observado. El uso de `StratifiedKFold` asegura que cada partición mantenga la proporción original de estudiantes que completan/no completan, evitando folds con muy pocos casos de la clase minoritaria.

## 2. Modelo de regresión con pipeline y ajuste de hiperparámetros (`cursos_futuros`)

Se usa `RandomForestRegressor`, optimizando también `n_estimators` y `max_depth` con `GridSearchCV`, usando `neg_mean_absolute_error` como criterio de selección.

In [5]:
from sklearn.ensemble import RandomForestRegressor

X_reg = df.drop(columns=['completo', 'cursos_futuros'])
y_reg = df['cursos_futuros']

pipe_reg = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(random_state=42)),
])

param_grid_reg = {
    'regressor__n_estimators': [100, 200],
    'regressor__max_depth': [3, 5, None],
}

grid_reg = GridSearchCV(
    pipe_reg, param_grid_reg, cv=5,
    scoring='neg_mean_absolute_error', n_jobs=-1,
)
grid_reg.fit(X_reg, y_reg)

print(f'Mejores parámetros (regresión): {grid_reg.best_params_}')
print(f'Mejor MAE en la búsqueda: {-grid_reg.best_score_:.4f}')

Mejores parámetros (regresión): {'regressor__max_depth': 5, 'regressor__n_estimators': 100}
Mejor MAE en la búsqueda: 0.6365


In [6]:
resultados_reg = cross_validate(
    grid_reg.best_estimator_, X_reg, y_reg, cv=5,
    scoring=['neg_mean_absolute_error', 'r2'],
)

mae_promedio = -resultados_reg['test_neg_mean_absolute_error'].mean()
r2_promedio = resultados_reg['test_r2'].mean()

print(f'MAE promedio (modelo optimizado): {mae_promedio:.4f} cursos')
print(f'R² promedio (modelo optimizado): {r2_promedio:.4f}')

MAE promedio (modelo optimizado): 0.6365 cursos
R² promedio (modelo optimizado): 0.6341


**Justificación técnica:** se limitó `max_depth` a valores moderados (3, 5, o sin límite) para explorar el trade-off entre capacidad del modelo y riesgo de sobreajuste; `n_estimators` más alto generalmente estabiliza las predicciones del bosque a costa de mayor tiempo de cómputo. `GridSearchCV` selecciona automáticamente, dentro de la grilla probada, la combinación con menor error absoluto medio en validación cruzada.

## 3. Reflexión integradora y análisis de desempeño

**Comparación de enfoques:** ambos modelos comparten la misma arquitectura general (Pipeline con `ColumnTransformer` para escalamiento/codificación + `RandomForest` optimizado con `GridSearchCV`), lo que permite aislar la comparación a la naturaleza de la variable objetivo: `completo` es binaria (clasificación, evaluada con accuracy y f1_macro) y `cursos_futuros` es continua (regresión, evaluada con MAE y R²). Usar el mismo tipo de algoritmo base (bosques aleatorios) en ambos casos facilita comparar el efecto del preprocesamiento y la optimización de hiperparámetros de forma más limpia.

**Impacto de la validación cruzada y el ajuste de hiperparámetros:** `GridSearchCV` evita elegir hiperparámetros "a ojo" y en su lugar los selecciona en base al desempeño promedio sobre múltiples particiones de los datos, reduciendo el riesgo de quedarse con una configuración que solo funciona bien por azar en una partición específica. El uso de `StratifiedKFold` en el caso de clasificación fue clave para no subestimar el error en la clase minoritaria de `completo`. En ambos modelos, la validación cruzada posterior a `GridSearchCV` confirma que el desempeño reportado no depende de una única partición train/test, sino que es un promedio robusto.

**Impacto del preprocesamiento:** el `ColumnTransformer` (escalamiento de variables numéricas + codificación one-hot de categóricas) dentro del mismo `Pipeline` que el modelo garantiza que el preprocesamiento se ajuste (fit) solo con los datos de entrenamiento de cada fold, evitando fuga de información (data leakage) desde el conjunto de validación hacia el entrenamiento — algo que ocurriría si se escalara o codificara el dataset completo antes de la validación cruzada.

**¿Cuál modelo es más confiable?** Con los resultados obtenidos (accuracy ≈ 0.66 y f1_macro ≈ 0.63 en clasificación; R² ≈ 0.63 y MAE ≈ 0.64 cursos en regresión), ambos modelos muestran un desempeño moderado y bastante similar entre sí — ninguno domina claramente al otro. El modelo de regresión tiene a su favor un R² razonablemente alto (explica ~63% de la variabilidad de `cursos_futuros`) con un error absoluto medio pequeño en términos absolutos (menos de 1 curso de diferencia en promedio); el modelo de clasificación, en cambio, se ve algo más afectado por el desbalance de clases (60%/40%), lo que explica que su f1_macro sea menor que su accuracy. En términos prácticos, el modelo de regresión parece levemente más confiable para la toma de decisiones de negocio en este dataset, aunque ambos se beneficiarían de más datos o de variables predictoras adicionales para mejorar su desempeño.

## 4. Conclusiones para audiencia no técnica

*(Sección agregada al incorporar este trabajo al portafolio, como mejora sobre la versión original entregada en el curso.)*

En palabras simples: se construyeron dos "predictores" a partir de los datos de una plataforma de formación continua. El primero intenta adivinar si un estudiante va a **terminar el curso en el que se inscribió**, y el segundo intenta estimar **cuántos cursos más va a tomar** en los próximos meses.

Ninguno de los dos modelos acierta siempre —ni podría hacerlo, porque el comportamiento de las personas depende de muchos factores que no están en estos datos—, pero ambos aciertan bastante más que si se adivinara al azar. El que estima cuántos cursos futuros tomará una persona es, de los dos, el que da resultados algo más confiables: en promedio se equivoca por menos de un curso de diferencia. El que predice si alguien completará el curso actual funciona razonablemente bien, aunque le cuesta un poco más detectar a quienes no lo van a completar, porque en los datos hay más ejemplos de estudiantes que sí terminan que de los que no.

¿Para qué sirve esto en la práctica? Un equipo de la plataforma podría usar el primer modelo para identificar a tiempo a estudiantes con riesgo de abandonar y ofrecerles apoyo adicional, y el segundo para anticipar la demanda de cursos futuros y planificar mejor la oferta académica.